In [2]:
import numpy as np
import pandas as pd
from gymnasium.utils.env_checker import check_env
import config as C
from download_data import FEATURE_COLS
from trading_env import TradingEnv


In [3]:
CRASH = (400,460)

def synthetic_panel(n_days = 800,seed= 0):
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range("2020-01-01", periods=n_days)
    drift = np.full(n_days, 0.001)
    drift[CRASH[0]:CRASH[1]] = -0.006
    cols, data = [], []
    for t in C.TICKERS:
        close = 100 * np.exp(np.cumsum(drift + rng.normal(0, 0.01, n_days)))
        for f in FEATURE_COLS:
            cols.append((t, f)); data.append(rng.normal(size=n_days))
        cols.append((t, "Close")); data.append(close)
    return pd.DataFrame(np.array(data).T,index=idx,columns=pd.MultiIndex.from_tuples(cols,names=["ticker","features"]))


In [4]:
def synthetic_news(T, n, D=16, p=0.3, seed=1):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(T, n, D)).astype(np.float32)
    x /= np.linalg.norm(x, axis=2, keepdims=True) * 4      # keep inside [-1, 1]
    x *= (rng.random((T, n, 1)) < p)                       # most steps have no news
    return x


In [5]:
def act(env, weights=None, cash=False):
    """Build a softmax-mode action: +1 logit for chosen slots, -1 elsewhere."""
    a = -np.ones(env.action_space.shape[0], np.float32)
    if cash:
        a[-1] = 1.0
    else:
        a[:env.n] = 1.0 if weights is None else np.where(weights, 1.0, -1.0)
    return a


In [6]:
def run(env, policy):
    env.reset(seed=0)
    total_r = total_cost = 0.0; done = False
    while not done:
        _, r, term, trunc, info = env.step(policy(env))
        total_r += r; total_cost += info["cost"]; done = term or trunc
    return info["equity"], total_r, total_cost


In [7]:
panel = synthetic_panel()

check_env(TradingEnv(panel),skip_render_check=True)


In [8]:
news = synthetic_news(len(panel), len(C.TICKERS))
env_n = TradingEnv(panel, news=news)
check_env(env_n, skip_render_check=True)


In [9]:
obs, _ = env_n.reset()
print("check_env OK (no news + news). obs keys:", list(obs), "| action dim:", env_n.action_space.shape[0])
print("news obs:", obs["news"].shape, "age range:", obs["news_age"].min(), "-", obs["news_age"].max())


check_env OK (no news + news). obs keys: ['market', 'portfolio', 'news', 'news_age'] | action dim: 7
news obs: (6, 16) age range: 0.0 - 0.4


In [10]:
def buy_hold(e):   return act(e)
def churner(e):    # flips 100% between two assets every step
    return act(e, np.arange(e.n) == (e.t % 2))
def oracle(e):     # knows the crash: sits in cash from close 399 to 459 (delay=1 -> decide earlier)
    return act(e, cash=True) if CRASH[0] - 2 <= e.t <= CRASH[1] - 3 else act(e)
def all_cash(e):   return act(e, cash=True)


In [11]:
print(f"\n{'policy':10s} {'equity':>8s} {'total reward':>13s} {'costs':>8s}")
res = {}
for name, pol in [("oracle", oracle), ("buy&hold", buy_hold), ("cash", all_cash), ("churner", churner)]:
    res[name] = run(TradingEnv(panel), pol)
    print(f"{name:10s} {res[name][0]:8.3f} {res[name][1]:13.1f} {res[name][2]:8.3f}")
assert res["oracle"][1] > res["buy&hold"][1] > res["churner"][1], "reward ordering wrong"
assert res["oracle"][0] > res["buy&hold"][0] > res["churner"][0], "equity ordering wrong"
print("Incentives OK: de-risking before a crash > holding through it > churning")



policy       equity  total reward    costs
oracle        1.857          53.5    0.004
buy&hold      1.340          18.7    0.001
cash          1.000           0.0    0.000
churner       0.045        -469.5    3.288
Incentives OK: de-risking before a crash > holding through it > churning


In [12]:
rng = np.random.default_rng(0)
e = TradingEnv(panel); e.reset(seed=0); tv = 0.0; done = False
base = act(e)
while not done:
    _, _, term, trunc, info = e.step(np.clip(base + rng.normal(0, 0.02, base.shape), -1, 1).astype(np.float32))
    tv += info["turnover"]; done = term or trunc
print(f"Deadband test: total turnover under jittery actions = {tv:.2f} over 800 steps (vs ~1000+ if it traded every step)")


Deadband test: total turnover under jittery actions = 3.46 over 800 steps (vs ~1000+ if it traded every step)


In [13]:
e = TradingEnv(panel); e.reset(seed=0); _, _, _, _, info = e.step(buy_hold(e))
print("aux_target shape:", info["aux_target"].shape, "mask:", info["aux_mask"])


aux_target shape: (6, 3) mask: [1. 1. 1.]
